In [ ]:
! pip install -q pypdf sentence-transformers faiss-cpu accelerate transformers scikit-learn numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 45.8 MB/s eta 0:00:00


In [ ]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)

In [ ]:
from google.colab import files

uploaded = files.upload()


In [ ]:
reader = PdfReader("Hybrid_Search_Practice.pdf")
text = "\n".join(page.extract_text() for page in reader.pages)
print(text)

In [ ]:
chunk_size = 50
words = text.split()
chunk_list = []
for i in range(0, len(words), chunk_size):
    chunk_list.append(" ".join(words[i:i+chunk_size]))
print(chunk_list)

In [ ]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
chunk_embeddings = embedding_model.encode(chunk_list)
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)

In [ ]:
question = "What is mean by Hybrid Search?"
question_embedding = embedding_model.encode([question])

distance, index_number = index.search(
    np.array(question_embedding),
    k=3
)

retrieved_chunks = []
print(index_number[0])

for idx in index_number[0]:
    retrieved_chunks.append(chunk_list[idx])

print(retrieved_chunks)


In [ ]:
tfidf_vectorizer = TfidfVectorizer() # Create empty TF-IDF model

#it performs two functions fit() and transforms. Fit() reads every chunks and later convert them into numbers
tfidf_matrix = tfidf_vectorizer.fit_transform(chunk_list)

question_tfidf = tfidf_vectorizer.transform([question])

similarity_scores = cosine_similarity(
    question_tfidf,
    tfidf_matrix
)

top_k = 3

top_indices = np.argsort(similarity_scores[0])[::-1][:top_k]
keyword_chunk = [chunk_list[rec] for rec in top_indices]

keyword_indices = top_indices.tolist()
semantic_indices = index_number[0].tolist()

print(keyword_indices,semantic_indices)
combined = semantic_indices + keyword_indices
hybrid_indices = list(dict.fromkeys(combined))

print(hybrid_indices)


In [ ]:
hybrid_chunks = []

for idx in hybrid_indices:
    hybrid_chunks.append(chunk_list[idx])

context = "\n\n".join(hybrid_chunks)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct"
)

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="auto"
)

chatbot = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer
)

In [ ]:
prompt = f"""
<|user|>

Use ONLY the context below.

Context:
{context}

Question:
{question}

If the answer is not present, reply exactly:

I couldn't find that information.

<|assistant|>
"""
response = chatbot(
    prompt,
    max_new_tokens=120,
    do_sample=False,
    return_full_text=False
)

answer = response[0]["generated_text"].strip()

print(answer)